READ SILVER_SALES

In [303]:
# 1. Required libraries

import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os

# Load environment variable

load_dotenv()

mysql_user = os.getenv("MYSQL_USER")
mysql_password = quote_plus(os.getenv("MYSQL_PASSWORD"))
mysql_host = os.getenv("MYSQL_HOST")
mysql_port = os.getenv("MYSQL_PORT")
mysql_database = os.getenv("MYSQL_DATABASE")

In [304]:
# 2. CONNECT MYSQL 

try:
    engine = create_engine(
        f"mysql+pymysql://{mysql_user}:{mysql_password}@"
        f"{mysql_host}:{mysql_port}/{mysql_database}"
        
    )

    # test connection
    with engine.connect() as connection:
        print("Successfully connected MYSQL !")


except Exception as e:
    print(f"Error: Couldn't connect to MYSQL {e}")

Successfully connected MYSQL !


In [305]:
# 3. READ SILVER

df = pd.read_sql(
    "SELECT * FROM silver_sales",
    engine
)

print("Silver data loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Silver data loaded successfully!
Rows: 9800
Columns: 22


4. CREATE THE DIMENSION DATAFRAMES

In [306]:
# 4.1 CUSTOMER DIMENSION

dim_customer = df[
    [
        "customer_id",
        "customer_name",
        "segment"
    ]
].drop_duplicates()

print("Customer dimension rows:", len(dim_customer))

Customer dimension rows: 793


In [307]:
# 4.2 PRODUCT DIMENSION

dim_product = df[
    [
        "product_id",
        "product_name",
        "category",
        "sub_category"
    ]
].drop_duplicates()

print("Product dimension rows:", len(dim_product))

Product dimension rows: 1893


In [308]:
# 4.3 LOCATION DIMENSION

dim_location = df[
    [
        "country",
        "city",
        "state",
        "postal_code",
        "region"
    ]
].drop_duplicates()

print("Location dimension rows:", len(dim_location))

Location dimension rows: 628


In [309]:
# 4.4 DATE DIMENSION

dim_date = df[
    [
        "order_date"
    ]
].drop_duplicates()

print("Date dimension rows:", len(dim_date))

Date dimension rows: 1230


In [313]:
# Convert the column to datetime first
dim_date["order_date"] = pd.to_datetime(dim_date["order_date"])

dim_date["year"] = dim_date["order_date"].dt.year
dim_date["month"] = dim_date["order_date"].dt.month
dim_date["month_name"] = dim_date["order_date"].dt.month_name()
dim_date["quarter"] = dim_date["order_date"].dt.quarter
dim_date["day"] = dim_date["order_date"].dt.day
dim_date["day_name"] = dim_date["order_date"].dt.day_name()

print(dim_date)

print(dim_date["order_date"].dtype)

     order_date  year  month month_name  quarter  day   day_name
0    2017-11-08  2017     11   November        4    8  Wednesday
2    2017-06-12  2017      6       June        2   12     Monday
3    2016-10-11  2016     10    October        4   11    Tuesday
5    2015-06-09  2015      6       June        2    9    Tuesday
12   2018-04-15  2018      4      April        2   15     Sunday
...         ...   ...    ...        ...      ...  ...        ...
9696 2015-06-10  2015      6       June        2   10  Wednesday
9750 2017-10-11  2017     10    October        4   11  Wednesday
9764 2015-06-18  2015      6       June        2   18   Thursday
9765 2018-02-28  2018      2   February        1   28  Wednesday
9785 2016-05-09  2016      5        May        2    9     Monday

[1230 rows x 7 columns]
datetime64[ns]


In [314]:
# surrogate/date key

# rename 

dim_date = dim_date.rename(
    columns = {"order_date": "full_date"}
)

print(dim_date.dtypes)

dim_date["date_key"] = (
    dim_date["full_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

print(dim_date.columns)

full_date     datetime64[ns]
year                   int32
month                  int32
month_name            object
quarter                int32
day                    int32
day_name              object
dtype: object
Index(['full_date', 'year', 'month', 'month_name', 'quarter', 'day',
       'day_name', 'date_key'],
      dtype='object')


In [315]:
print(dim_date.head())
print(dim_date.columns)

print(dim_date.shape)
print(dim_date["date_key"].is_unique)

    full_date  year  month month_name  quarter  day   day_name  date_key
0  2017-11-08  2017     11   November        4    8  Wednesday  20171108
2  2017-06-12  2017      6       June        2   12     Monday  20170612
3  2016-10-11  2016     10    October        4   11    Tuesday  20161011
5  2015-06-09  2015      6       June        2    9    Tuesday  20150609
12 2018-04-15  2018      4      April        2   15     Sunday  20180415
Index(['full_date', 'year', 'month', 'month_name', 'quarter', 'day',
       'day_name', 'date_key'],
      dtype='object')
(1230, 8)
True


5. Inspection of DIMENSION 

5.1 DIM_CUSTOMER

In [316]:
print("========== DIM CUSTOMER ==========")

print("Rows:", len(dim_customer))
print("Columns:", len(dim_customer.columns))

print("\nColumns:")
print(dim_customer.columns.tolist())

print("\nFirst 5 rows:")
print(dim_customer.head())

========== DIM CUSTOMER ==========
Rows: 793
Columns: 3

Columns:
['customer_id', 'customer_name', 'segment']

First 5 rows:
   customer_id    customer_name    segment
0     CG-12520      Claire Gute   Consumer
2     DV-13045  Darrin Van Huff  Corporate
3     SO-20335   Sean O'Donnell   Consumer
5     BH-11710  Brosina Hoffman   Consumer
12    AA-10480     Andrew Allen   Consumer


In [317]:
print("\nDuplicate customer_ids:",
      dim_customer["customer_id"].duplicated().sum())

print("Unique customer_ids:",
      dim_customer["customer_id"].nunique())


Duplicate customer_ids: 0
Unique customer_ids: 793


In [318]:
assert dim_customer["customer_id"].nunique() == len(dim_customer)

In [319]:
print("\nMissing values:")
print(dim_customer.isna().sum())

assert dim_customer["customer_id"].notna().all()
assert dim_customer["customer_name"].notna().all()
assert dim_customer["segment"].notna().all()

print("Customer dimension validation passed!")


Missing values:
customer_id      0
customer_name    0
segment          0
dtype: int64
Customer dimension validation passed!


5.2 DIM PRODUCT

In [320]:
print("\n========== DIM PRODUCT ==========")

print("Rows:", len(dim_product))
print("Columns:", len(dim_product.columns))

print("\nColumns:")
print(dim_product.columns.tolist())

print("\nFirst 5 rows:")
print(dim_product.head())


========== DIM PRODUCT ==========
Rows: 1893
Columns: 4

Columns:
['product_id', 'product_name', 'category', 'sub_category']

First 5 rows:
        product_id                                       product_name  \
0  FUR-BO-10001798                  Bush Somerset Collection Bookcase   
1  FUR-CH-10000454  Hon Deluxe Fabric Upholstered Stacking Chairs,...   
2  OFF-LA-10000240  Self-Adhesive Address Labels for Typewriters b...   
3  FUR-TA-10000577      Bretford CR4500 Series Slim Rectangular Table   
4  OFF-ST-10000760                     Eldon Fold 'N Roll Cart System   

          category sub_category  
0        Furniture    Bookcases  
1        Furniture       Chairs  
2  Office Supplies       Labels  
3        Furniture       Tables  
4  Office Supplies      Storage  


In [321]:
print("\nDuplicate product_ids:",
      dim_product["product_id"].duplicated().sum())

print("Unique product_ids:",
      dim_product["product_id"].nunique())


Duplicate product_ids: 32
Unique product_ids: 1861


In [322]:
duplicate_product_ids = (
    dim_product[
        dim_product["product_id"].duplicated(keep=False)
    ]
    .sort_values("product_id")
)

print(duplicate_product_ids) 

           product_id                                       product_name  \
2471  FUR-BO-10002213   Sauder Forest Hills Library, Woodland Oak Finish   
2115  FUR-BO-10002213              DMI Eclipse Executive Suite Bookcases   
66    FUR-CH-10001146        Global Value Mid-Back Manager's Chair, Gray   
128   FUR-CH-10001146                           Global Task Chair, Black   
1459  FUR-FU-10001473                            DAX Wood Document Frame   
...               ...                                                ...   
1219  TEC-PH-10002200                              Samsung Galaxy Note 2   
2596  TEC-PH-10002310  Plantronics Calisto P620-M USB Wireless Speake...   
1378  TEC-PH-10002310                 Panasonic KX T7731-B Digital phone   
922   TEC-PH-10004531      OtterBox Commuter Series Case - iPhone 5 & 5s   
2713  TEC-PH-10004531                                        AT&T CL2909   

        category sub_category  
2471   Furniture    Bookcases  
2115   Furniture    Boo

In [323]:
print(
    duplicate_product_ids["product_id"]
    .value_counts()
)

product_id
FUR-BO-10002213    2
FUR-CH-10001146    2
TEC-PH-10002310    2
TEC-PH-10002200    2
TEC-PH-10001795    2
TEC-PH-10001530    2
TEC-MA-10001148    2
TEC-AC-10003832    2
TEC-AC-10002550    2
TEC-AC-10002049    2
OFF-ST-10004950    2
OFF-ST-10001228    2
OFF-PA-10003022    2
OFF-PA-10002377    2
OFF-PA-10002195    2
OFF-PA-10001970    2
OFF-PA-10001166    2
OFF-PA-10000659    2
OFF-PA-10000477    2
OFF-PA-10000357    2
OFF-BI-10004654    2
OFF-BI-10004632    2
OFF-BI-10002026    2
OFF-AR-10001149    2
OFF-AP-10000576    2
FUR-FU-10004864    2
FUR-FU-10004848    2
FUR-FU-10004270    2
FUR-FU-10004091    2
FUR-FU-10004017    2
FUR-FU-10001473    2
TEC-PH-10004531    2
Name: count, dtype: int64


In [324]:
duplicate_products = (
    dim_product[
        dim_product["product_id"].duplicated(keep=False)
    ]
    .sort_values("product_id")
)

print(
    duplicate_products[
        ["product_id", "product_name", "category", "sub_category"]
    ].to_string(index=False)
)

     product_id                                                                                    product_name        category sub_category
FUR-BO-10002213                                                Sauder Forest Hills Library, Woodland Oak Finish       Furniture    Bookcases
FUR-BO-10002213                                                           DMI Eclipse Executive Suite Bookcases       Furniture    Bookcases
FUR-CH-10001146                                                     Global Value Mid-Back Manager's Chair, Gray       Furniture       Chairs
FUR-CH-10001146                                                                        Global Task Chair, Black       Furniture       Chairs
FUR-FU-10001473                                                                         DAX Wood Document Frame       Furniture  Furnishings
FUR-FU-10001473                                          Eldon Executive Woodline II Desk Accessories, Mahogany       Furniture  Furnishings
FUR-FU-100040

In [325]:
conflicting_products = (
    dim_product
    .groupby("product_id")
    .agg(
        product_name_count=("product_name", "nunique"),
        category_count=("category", "nunique"),
        sub_category_count=("sub_category", "nunique")
    )
)

print(
    conflicting_products[
        (conflicting_products["product_name_count"] > 1) |
        (conflicting_products["category_count"] > 1) |
        (conflicting_products["sub_category_count"] > 1)
    ]
)

                 product_name_count  category_count  sub_category_count
product_id                                                             
FUR-BO-10002213                   2               1                   1
FUR-CH-10001146                   2               1                   1
FUR-FU-10001473                   2               1                   1
FUR-FU-10004017                   2               1                   1
FUR-FU-10004091                   2               1                   1
FUR-FU-10004270                   2               1                   1
FUR-FU-10004848                   2               1                   1
FUR-FU-10004864                   2               1                   1
OFF-AP-10000576                   2               1                   1
OFF-AR-10001149                   2               1                   1
OFF-BI-10002026                   2               1                   1
OFF-BI-10004632                   2               1             

Multiple product names for same `product_id` |

32 product IDs have 2 distinct product names | 

Product dimension should have one record per product ID |
 
Investigate source inconsistency; preserve source records until business rule is established |


In [326]:
print("\nMissing values:")
print(dim_product.isna().sum())


Missing values:
product_id      0
product_name    0
category        0
sub_category    0
dtype: int64


In [327]:
assert dim_product["product_id"].notna().all()
assert dim_product["product_name"].notna().all()
assert dim_product["category"].notna().all()
assert dim_product["sub_category"].notna().all()

print("\nDistinct product records:", len(dim_product))

print("Product dimension validation passed!")


Distinct product records: 1893
Product dimension validation passed!


In [328]:
# explicitly detecting and documenting the anology 

conflicting_product_ids = (
    dim_product
    .groupby("product_id")["product_name"]
    .nunique()
)

conflicting_product_ids = conflicting_product_ids[
    conflicting_product_ids > 1
]

print("\nProduct IDs with multiple product names:",
      len(conflicting_product_ids))

assert len(conflicting_product_ids) == 32


Product IDs with multiple product names: 32


5.3 DIMENSION LOCATION

In [329]:
print("\n========== DIM LOCATION ==========")

print("Rows:", len(dim_location))
print("Columns:", len(dim_location.columns))

print("\nColumns:")
print(dim_location.columns.tolist())

print("\nFirst 5 rows:")
print(dim_location.head())


========== DIM LOCATION ==========
Rows: 628
Columns: 5

Columns:
['country', 'city', 'state', 'postal_code', 'region']

First 5 rows:
          country             city           state postal_code region
0   United States        Henderson        Kentucky       42420  South
2   United States      Los Angeles      California       90036   West
3   United States  Fort Lauderdale         Florida       33311  South
5   United States      Los Angeles      California       90032   West
12  United States          Concord  North Carolina       28027  South


In [330]:
# check duplicates with all five attributes

location_columns = [
    "country",
    "city",
    "state",
    "postal_code",
    "region"
]

duplicate_locations = dim_location.duplicated(
    subset=location_columns
).sum()

print("Duplicate locations:", duplicate_locations)

Duplicate locations: 0


In [331]:
assert duplicate_locations == 0

In [332]:
# check nulls

print("\nMissing values:")
print(dim_location.isna().sum())


Missing values:
country        0
city           0
state          0
postal_code    1
region         0
dtype: int64


In [333]:
print(
    "Missing postal codes:",
    dim_location["postal_code"].isna().sum()
)

Missing postal codes: 1


In [334]:
assert dim_location["country"].notna().all()
assert dim_location["city"].notna().all()
assert dim_location["state"].notna().all()
assert dim_location["region"].notna().all()

print("Location dimension validation passed!")

Location dimension validation passed!


5.4 DIMENSION DATE

In [335]:
print("\n========== DIM DATE ==========")

print("Rows:", len(dim_date))
print("Columns:", len(dim_date.columns))

print("\nColumns:")
print(dim_date.columns.tolist())

print("\nFirst 5 rows:")
print(dim_date.head())


========== DIM DATE ==========
Rows: 1230
Columns: 8

Columns:
['full_date', 'year', 'month', 'month_name', 'quarter', 'day', 'day_name', 'date_key']

First 5 rows:
    full_date  year  month month_name  quarter  day   day_name  date_key
0  2017-11-08  2017     11   November        4    8  Wednesday  20171108
2  2017-06-12  2017      6       June        2   12     Monday  20170612
3  2016-10-11  2016     10    October        4   11    Tuesday  20161011
5  2015-06-09  2015      6       June        2    9    Tuesday  20150609
12 2018-04-15  2018      4      April        2   15     Sunday  20180415


In [336]:
# Check unique dates

print("\nDuplicate date_keys:",
      dim_date["date_key"].duplicated().sum())

print("Unique date_keys:",
      dim_date["date_key"].nunique())


Duplicate date_keys: 0
Unique date_keys: 1230


In [337]:
assert dim_date["date_key"].nunique() == len(dim_date)

In [338]:
# check full date uniqueness

print(
    "Duplicate full_dates:",
    dim_date["full_date"].duplicated().sum()
)

Duplicate full_dates: 0


In [339]:
assert dim_date["full_date"].nunique() == len(dim_date)

In [340]:
print("\nMissing values:")
print(dim_date.isna().sum())


Missing values:
full_date     0
year          0
month         0
month_name    0
quarter       0
day           0
day_name      0
date_key      0
dtype: int64


In [341]:
assert dim_date.isna().sum().sum() == 0

In [342]:
expected_date_keys = (
    dim_date["full_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

assert (
    dim_date["date_key"] == expected_date_keys
).all()

print("Date key consistency passed!")

Date key consistency passed!


FINAL VALIDATION CHECK

In [343]:
print("\n========================================")
print("       GOLD DIMENSION VALIDATION")
print("========================================")

print(f"Customer rows : {len(dim_customer)}")
print(f"Product rows  : {len(dim_product)}")
print(f"Location rows : {len(dim_location)}")
print(f"Date rows     : {len(dim_date)}")

print("\nCustomer unique IDs:",
      dim_customer["customer_id"].nunique())

print("Product unique IDs:",
      dim_product["product_id"].nunique())

print("Duplicate locations:",
      dim_location.duplicated(
          subset=location_columns
      ).sum())

print("Date unique keys:",
      dim_date["date_key"].nunique())

print("\nAll Gold dimension validations passed!")


       GOLD DIMENSION VALIDATION
Customer rows : 793
Product rows  : 1893
Location rows : 628
Date rows     : 1230

Customer unique IDs: 793
Product unique IDs: 1861
Duplicate locations: 0
Date unique keys: 1230

All Gold dimension validations passed!


# 6. Loading dataframes into mysql tables


In [344]:
print("\n========== LOADING GOLD DIMENSIONS ==========")

# Customer Table 

dim_customer.to_sql(
    name="dim_customer",
    con=engine,
    if_exists='append',
    index=False
)

print("Customer dimension loaded successfully!")


========== LOADING GOLD DIMENSIONS ==========
Customer dimension loaded successfully!


In [345]:
# Product Table

dim_product.to_sql(
    name="dim_product",
    con=engine,
    if_exists="append",
    index=False
)

print("Product dimension loaded successfully!")

Product dimension loaded successfully!


In [346]:
# Location Table

dim_location.to_sql(
    name = "dim_location",
    con=engine,
    if_exists = "append",
    index=False
)

print("Product dimension loaded successfully!")

Product dimension loaded successfully!


In [347]:
# Date Table

dim_date.to_sql(
    name="dim_date",
    con=engine,
    if_exists = "append",
    index=False
)

print("Date dimension loaded successfully!")

Date dimension loaded successfully!


FACT TABLE

In [ ]:
dim_customer_db = pd.read_sql(
    """
    SELECT
        customer_key,
        customer_id
    FROM dim_customer
    """,
    engine
)

# each row identifies uniquely 

In [351]:
# 32 product IDs have multiple product names, we CANNOT join only on product_id.

dim_product_db = pd.read_sql(
    """
    SELECT 
        product_key,
        product_id,
        product_name,
        category,
        sub_category
    FROM dim_product
    """,
    engine
)

In [352]:
dim_location_db = pd.read_sql(
    """
    SELECT
        location_key,
        country,
        city,
        state,
        postal_code,
        region
    FROM dim_location
    """,
    engine
)

In [353]:
dim_date_db = pd.read_sql(
    """
    SELECT
        date_key,
        full_date
    FROM dim_date
    """,
    engine
)

### Mapping dimensions key to fact table


In [ ]:
# Map customer_key

df = df.merge(
    dim_customer_db,
    on = "customer_id",
    how = "left"
)



In [356]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   row_id               9800 non-null   int64         
 1   order_id             9800 non-null   object        
 2   order_date           9800 non-null   object        
 3   ship_date            9800 non-null   object        
 4   ship_mode            9800 non-null   object        
 5   customer_id          9800 non-null   object        
 6   customer_name        9800 non-null   object        
 7   segment              9800 non-null   object        
 8   country              9800 non-null   object        
 9   city                 9800 non-null   object        
 10  state                9800 non-null   object        
 11  postal_code          9789 non-null   object        
 12  region               9800 non-null   object        
 13  product_id           9800 non-nul

In [358]:
print(df.head(3))

   row_id        order_id  order_date   ship_date     ship_mode customer_id  \
0       1  CA-2017-152156  2017-11-08  2017-11-11  Second Class    CG-12520   
1       2  CA-2017-152156  2017-11-08  2017-11-11  Second Class    CG-12520   
2       3  CA-2017-138688  2017-06-12  2017-06-16  Second Class    DV-13045   

     customer_name    segment        country         city  ...  \
0      Claire Gute   Consumer  United States    Henderson  ...   
1      Claire Gute   Consumer  United States    Henderson  ...   
2  Darrin Van Huff  Corporate  United States  Los Angeles  ...   

        product_id         category sub_category  \
0  FUR-BO-10001798        Furniture    Bookcases   
1  FUR-CH-10000454        Furniture       Chairs   
2  OFF-LA-10000240  Office Supplies       Labels   

                                        product_name   sales order_year  \
0                  Bush Somerset Collection Bookcase  261.96       2017   
1  Hon Deluxe Fabric Upholstered Stacking Chairs,...  731.9

In [361]:
print("How many rows failed to match with a customer record during the merge.")
print(df["customer_key"].isna().sum())

How many rows failed to match with a customer record during the merge.
0


In [362]:
assert df["customer_key"].notna().all()

In [363]:
# Mapping Product_key

# product_id alone is Not unique, earlier we identified that 32 product id values map to 2 different product name.

product_join_columns = [
    "product_id",
    "product_name",
    "category",
    "sub_category"
]

df = df.merge(
    dim_product_db,
    on = product_join_columns,
    how = "left"
)

print("Successfully merged!")

Successfully merged!


In [364]:
print(df.head(3))

   row_id        order_id  order_date   ship_date     ship_mode customer_id  \
0       1  CA-2017-152156  2017-11-08  2017-11-11  Second Class    CG-12520   
1       2  CA-2017-152156  2017-11-08  2017-11-11  Second Class    CG-12520   
2       3  CA-2017-138688  2017-06-12  2017-06-16  Second Class    DV-13045   

     customer_name    segment        country         city  ...  \
0      Claire Gute   Consumer  United States    Henderson  ...   
1      Claire Gute   Consumer  United States    Henderson  ...   
2  Darrin Van Huff  Corporate  United States  Los Angeles  ...   

          category sub_category  \
0        Furniture    Bookcases   
1        Furniture       Chairs   
2  Office Supplies       Labels   

                                        product_name   sales order_year  \
0                  Bush Somerset Collection Bookcase  261.96       2017   
1  Hon Deluxe Fabric Upholstered Stacking Chairs,...  731.94       2017   
2  Self-Adhesive Address Labels for Typewriters b...

In [365]:
# Mapping date_key

df = df.merge(
    dim_date_db,
    left_on = "order_date",
    right_on = "full_date",
    how = "left"
)

print("Successfully merged!")

Successfully merged!


In [366]:
df.head(5)

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,product_name,sales,order_year,order_month,order_day,processed_timestamp,customer_key,product_key,date_key,full_date
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bush Somerset Collection Bookcase,261.96,2017,11,8,2026-09-09 19:04:03,1,1,20171108,2017-11-08
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,2017,11,8,2026-09-09 19:04:03,1,2,20171108,2017-11-08
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Self-Adhesive Address Labels for Typewriters b...,14.62,2017,6,12,2026-09-09 19:04:03,2,3,20170612,2017-06-12
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Bretford CR4500 Series Slim Rectangular Table,957.58,2016,10,11,2026-09-09 19:04:03,3,4,20161011,2016-10-11
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Eldon Fold 'N Roll Cart System,22.37,2016,10,11,2026-09-09 19:04:03,3,5,20161011,2016-10-11


In [367]:
print(
    "Missing date keys:",
    df["date_key"].isna().sum()
)

assert df["date_key"].notna().all()

Missing date keys: 0


In [368]:
# Mapping Location_key

location_columns_join = [
    "country",
    "city",
    "state",
    "postal_code",
    "region"
]

In [ ]:
df = df.merge(
    dim_location_db,
    on = location_columns_join,
    how = "left"
)

In [370]:
print(
    "Missing location keys:",
    df["location_key"].isna().sum()
)

Missing location keys: 0


In [371]:
df.head(5)

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,sales,order_year,order_month,order_day,processed_timestamp,customer_key,product_key,date_key,full_date,location_key
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,261.96,2017,11,8,2026-09-09 19:04:03,1,1,20171108,2017-11-08,1
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,731.94,2017,11,8,2026-09-09 19:04:03,1,2,20171108,2017-11-08,1
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,14.62,2017,6,12,2026-09-09 19:04:03,2,3,20170612,2017-06-12,2
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,957.58,2016,10,11,2026-09-09 19:04:03,3,4,20161011,2016-10-11,3
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,22.37,2016,10,11,2026-09-09 19:04:03,3,5,20161011,2016-10-11,3


In [372]:
assert df["location_key"].notna().all()

#### CREATE THE FACT DATAFRAME

In [377]:
fact_sales = df[
    [
        "row_id",
        "order_id",
        "date_key",
        "customer_key",
        "product_key",
        "location_key",
        "ship_mode",
        "ship_date",
        "sales"
    ]
].copy()

# .copy() - allocates fact_sales dataframe new memory location

In [380]:
print("\n========== FACT SALES ==========")

print("Rows:", len(fact_sales))
print("Columns:", len(fact_sales.columns))

print("\nColumns:")
print(fact_sales.columns.tolist())

print("\nFirst 5 rows:")
fact_sales.head(10)


========== FACT SALES ==========
Rows: 9800
Columns: 9

Columns:
['row_id', 'order_id', 'date_key', 'customer_key', 'product_key', 'location_key', 'ship_mode', 'ship_date', 'sales']

First 5 rows:


,row_id,order_id,date_key,customer_key,product_key,location_key,ship_mode,ship_date,sales
0,1,CA-2017-152156,20171108,1,1,1,Second Class,2017-11-11,261.96
1,2,CA-2017-152156,20171108,1,2,1,Second Class,2017-11-11,731.94
2,3,CA-2017-138688,20170612,2,3,2,Second Class,2017-06-16,14.62
3,4,US-2016-108966,20161011,3,4,3,Standard Class,2016-10-18,957.58
4,5,US-2016-108966,20161011,3,5,3,Standard Class,2016-10-18,22.37
5,6,CA-2015-115812,20150609,4,6,4,Standard Class,2015-06-14,48.86
6,7,CA-2015-115812,20150609,4,7,4,Standard Class,2015-06-14,7.28
7,8,CA-2015-115812,20150609,4,8,4,Standard Class,2015-06-14,907.15
8,9,CA-2015-115812,20150609,4,9,4,Standard Class,2015-06-14,18.50
9,10,CA-2015-115812,20150609,4,10,4,Standard Class,2015-06-14,114.90


In [381]:
assert len(fact_sales) == len(df)

In [382]:
assert fact_sales["row_id"].nunique() == len(fact_sales)

In [383]:
assert fact_sales["date_key"].notna().all()
assert fact_sales["customer_key"].notna().all()
assert fact_sales["product_key"].notna().all()
assert fact_sales["location_key"].notna().all()

In [384]:
assert (fact_sales["sales"] > 0).all()

In [385]:
print(
    "Duplicate fact rows:",
    fact_sales.duplicated().sum()
)

Duplicate fact rows: 0


In [387]:
fact_sales.to_sql(
    name="fact_sales",
    con=engine,
    if_exists="append",
    index=False
)

print("Fact sales loaded successfully!")

Fact sales loaded successfully!
